In [25]:
import torch

data = torch.load("/home/ki/evo-algo-gtsrb-loss-landscape-prob.pt", weights_only=False)
reduced_embeddings = data["embeddings"]
df = data["dataframe"]

In [26]:
data.keys()

dict_keys(['embeddings', 'dataframe'])

In [27]:
reduced_embeddings = data["embeddings"]
df = data["dataframe"]

In [29]:
df

,signature,generation,individual_idx,fitness,size,complexity,auroc,aupr-id,aupr-ood,fpr95
0,((color=blue -> color=blue) -> (label=animals ...,28,9328,0.487831,2,8,0.487911,0.388712,0.770106,0.965075
1,((color=blue -> color=blue) -> (label=bend or ...,97,8612,0.514430,1,7,0.514500,0.538867,0.782280,0.937040
2,((color=blue -> color=blue) -> (label=go_left ...,24,4962,0.499735,1,7,0.499805,0.313677,0.786192,1.000000
3,((color=blue -> color=blue) -> (label=keep_lef...,24,7017,0.486301,2,8,0.486381,0.346772,0.775295,0.992160
4,((color=blue -> color=blue) -> (label=no_traff...,72,3899,0.504217,2,8,0.504297,0.570505,0.786466,0.987170
...,...,...,...,...,...,...,...,...,...,...
158125,shape=octagon;shape=square,3,1959,0.499980,2,2,0.500000,0.713677,0.786323,1.000000
158126,shape=octagon;shape=triangle,4,7278,0.499980,2,2,0.500000,0.713677,0.786323,1.000000
158127,shape=square,2,1374,0.506531,1,1,0.506541,0.492781,0.779143,0.947731
158128,shape=square;shape=triangle,3,2133,0.499980,2,2,0.500000,0.713677,0.786323,1.000000


In [19]:
# import plotly.express as px
# import pandas as pd
# import numpy as np
#
# # Example Data
# df = pd.DataFrame({
#     'x': reduced_embeddings[:, 0],
#     'y': reduced_embeddings[:, 1],
#     'auroc': df.auroc,
#     'tree_name': df.tree_name  # List of strings for each tree embedding
# })
#
# # Find max AUROC point
# max_idx = df['auroc'].idxmax()
# df['highlight'] = ['red' if i == max_idx else 'blue' for i in range(len(df))]
#
# # Create interactive scatter plot
# fig = px.scatter(
#     df, x='x', y='y', color='auroc', hover_data=['tree_name'], color_continuous_scale='viridis'
# )
#
# # Highlight best AUROC point
# fig.add_trace(px.scatter(df.iloc[[max_idx]], x='x', y='y').data[0])
#
# fig.update_layout(title="Interactive t-SNE of Tree Embeddings", coloraxis_colorbar_title="AUROC")
#
# fig.write_html("/home/ki/tsne_tree_embeddings.html")
#
# # Show interactive plot
# fig.show()

In [9]:
df

,x,y,auroc,tree_name,highlight
0,-37.037586,-52.214417,0.487911,((color=blue -> color=blue) -> (label=animals ...,blue
1,10.213752,47.203667,0.514500,((color=blue -> color=blue) -> (label=bend or ...,blue
2,10.404312,-18.905256,0.499805,((color=blue -> color=blue) -> (label=go_left ...,blue
3,11.405379,-83.593086,0.486381,((color=blue -> color=blue) -> (label=keep_lef...,blue
4,-0.282837,-70.311707,0.504297,((color=blue -> color=blue) -> (label=no_traff...,blue
...,...,...,...,...,...
158125,-6.846922,-45.185047,0.500000,shape=octagon;shape=square,blue
158126,6.810673,-46.601242,0.500000,shape=octagon;shape=triangle,blue
158127,48.996746,38.440956,0.506541,shape=square,blue
158128,13.110745,-43.648613,0.500000,shape=square;shape=triangle,blue


In [21]:
from bokeh.plotting import figure, output_file, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.plotting import output_file
import pandas as pd
from bokeh.palettes import Viridis256
from bokeh.transform import linear_cmap
import numpy as np


# Define color mapping for AUROC values
color_mapper = linear_cmap(
    field_name="auroc",
    palette=Viridis256,
    low=df["auroc"].min(),
    high=df["auroc"].max(),
)


# Prepare Data
df = pd.DataFrame(
    {
        "x": reduced_embeddings[:, 0],
        "y": reduced_embeddings[:, 1],
        "auroc": df.auroc,
        "tree_name": df.tree_name.apply(
            lambda x: "\n".join(x.split(";"))
        ),  # List of strings for each tree embedding
    }
)

# Identify the highest AUROC point
max_idx = df["auroc"].idxmax()

# Define ColumnDataSource for Bokeh
source = ColumnDataSource(df)

# Create figure
p = figure(
    title="Interactive t-SNE of Tree Embeddings",
    width=1000,
    height=1000,
    tools="pan,wheel_zoom,reset",
)

# Scatter plot
p.scatter(
    x="x",
    y="y",
    source=source,
    size=2,
    alpha=0.6,
    color=color_mapper,
)

# Highlight the best AUROC point in red
# p.scatter(
#     x=[df.loc[max_idx, 'x']],
#     y=[df.loc[max_idx, 'y']],
#     size=10, color="red"
# )

# Add Hover Tool
hover = HoverTool()
hover.tooltips = [("Tree Name", "@tree_name"), ("AUROC", "@auroc")]
p.add_tools(hover)

# Output to an HTML file
output_file("tsne_tree_embeddings.html")
output_file(filename="/home/ki/tsne_tree_embeddings_bokeh.html")

print("Exported to tsne_tree_embeddings.html")

# Show interactive plot
show(p)

Exported to tsne_tree_embeddings.html


In [49]:
from bokeh.plotting import figure, output_file, show
from bokeh.models import ColumnDataSource, HoverTool, ColorBar
from bokeh.palettes import Viridis256
from bokeh.transform import linear_cmap
from bokeh.layouts import column
import pandas as pd
import torch
from numpy import random

data = torch.load("/home/ki/evo-algo-gtsrb-loss-landscape-prob.pt", weights_only=False)
reduced_embeddings = data["embeddings"]
df = data["dataframe"]

idx = random.permutation(np.arange(len(df)))[:100000]

# Define color mapping for AUROC values
color_mapper = linear_cmap(
    field_name="auroc",
    palette=Viridis256,
    low=df["auroc"].min(),
    high=df["auroc"].max(),
)

# Prepare Data
df = pd.DataFrame(
    {
        "x": reduced_embeddings[idx, 0],
        "y": reduced_embeddings[idx, 1],
        "auroc": df.iloc[idx].auroc,
        "tree_name": df.iloc[idx].signature.apply(
            lambda x: "<ul>"
            + "".join(f"<li>{item}</li>" for item in x.split(";"))
            + "</ul>"
        ),
    }
)

# Identify the highest AUROC point
max_idx = df["auroc"].idxmax()

# Define ColumnDataSource for Bokeh
source = ColumnDataSource(df)

# Create figure with full-screen scaling
p = figure(
    title="Interactive t-SNE of Tree Embeddings",
    tools="pan,wheel_zoom,reset",
    sizing_mode="stretch_both",  # Automatically fills available screen space
)

# Set initial x and y range
p.x_range.start = -200
p.x_range.end = 200
p.y_range.start = -200
p.y_range.end = 200

# Scatter plot
p.scatter(x="x", y="y", source=source, size=3, alpha=0.6, color=color_mapper)

# Add Hover Tool (now with HTML rendering)
hover = HoverTool()
hover.tooltips = [
    ("Knowledge Base", "@tree_name{safe}"),  # {safe} enables HTML rendering
    ("AUROC", "@auroc{0.000000}"),
]
p.add_tools(hover)

# Add color bar
color_bar = ColorBar(color_mapper=color_mapper["transform"], width=8, location=(0, 0))
p.add_layout(color_bar, "right")

# Layout to allow full-screen scaling
layout = column(p, sizing_mode="stretch_both")

# Output to an HTML file
output_file("/home/ki/tsne_tree_embeddings_bokeh_prob.html")


# Show interactive plot
show(layout)

In [31]:
df.signature.apply(
    lambda x: "<ul>" + "".join(f"<li>{item}</li>" for item in x.split(";")) + "</ul>"
)

0         <ul><li>((color=blue -> color=blue) -> (label=...
1         <ul><li>((color=blue -> color=blue) -> (label=...
2         <ul><li>((color=blue -> color=blue) -> (label=...
3         <ul><li>((color=blue -> color=blue) -> (label=...
4         <ul><li>((color=blue -> color=blue) -> (label=...
                                ...                        
158125    <ul><li>shape=octagon</li><li>shape=square</li...
158126    <ul><li>shape=octagon</li><li>shape=triangle</...
158127                       <ul><li>shape=square</li></ul>
158128    <ul><li>shape=square</li><li>shape=triangle</l...
158129                     <ul><li>shape=triangle</li></ul>
Name: signature, Length: 158130, dtype: object